# Project 8 – Dashboard (OSS4008A)
**Notebook 02:** the web dashboard for Project 8, laid out from the course's *Draft for the project UI schema*:
AI assistant / analysis panel on the left, interactive map on the right, project buttons along the bottom.

**How to run:** Runtime → Change runtime type → **T4 GPU** (the plane detector runs much faster on a GPU), then Runtime → Run all.
Allow Google Drive when asked (the trained model is stored there), then click the link printed by the last cell.
The link works while this Colab session is open; every new session gives a new link.

**Data:** OurAirports (public domain), cleaned in notebook 01 and read from the course GitHub repository;
Ontario Imagery Web Map Service, Land Information Ontario (Open Government Licence – Ontario).

**AI:** the plane detector trained in notebook 04 (`plane_detector.pt` in Google Drive). Draw a box around an Ontario airport apron to detect planes.

In [ ]:
# Cell 1 – Install Streamlit (web app framework), folium (maps) and streamlit-folium (map ↔ app link)
!pip install -q streamlit streamlit-folium folium

In [ ]:
# Cell 1b – Connect Google Drive (the app loads the trained plane detector from there)
import os, torch
from google.colab import drive
drive.mount('/content/drive')
MODEL = '/content/drive/MyDrive/oss4008a_planes/plane_detector.pt'
print('Plane detector found ✅' if os.path.exists(MODEL) else '❌ plane_detector.pt not found: run notebook 04 first')
print('GPU available ✅' if torch.cuda.is_available() else '⚠️ No GPU: detection works but is slower (Runtime → Change runtime type → T4 GPU)')

## The app
The next cell **saves** the whole app as `app.py` (`%%writefile` writes the cell to a file instead of running it).
To change the app, edit this cell, then run Cells 2, 3 and 4 again. The link from Cell 5 stays the same.

In [ ]:
%%writefile app.py
# app.py – OSS4008A shared dashboard, Project 8: Airports and airplanes in Canada
# Layout follows the course's "Draft for the project UI schema":
#   left  = AI assistant / data analysis panel
#   right = interactive map (zoom, click a point for info, draw a box for an area summary)
#   bottom = one button per class project (only Project 8 is active in this app)
# Runs on Google Colab with Streamlit. All data is open:
#   - OurAirports (public domain), cleaned in notebook 01 and stored in the course GitHub repo
#   - Ontario Imagery Web Map Service, Land Information Ontario (Open Government Licence – Ontario)
# AI: a Faster R-CNN plane detector fine-tuned in notebook 04 on hand-labelled Ontario airport chips.
#   Draw a box on an Ontario airport → the app downloads the imagery, runs the model and draws the planes.

import copy                                  # copy.deepcopy gives each page run a fresh copy of the saved map
import io, math
from concurrent.futures import ThreadPoolExecutor   # download imagery tiles in parallel
import numpy as np
import requests
from PIL import Image
import pandas as pd
import streamlit as st
import folium
from folium.plugins import Draw              # adds the "draw a rectangle" tool to the map
from streamlit_folium import st_folium       # shows a folium map in Streamlit and returns what the user drew

st.set_page_config(page_title="OSS4008A Remote Sensing Dashboard", layout="wide")

# ---------------------------------------------------------------- data and settings
# Cleaned airport table from notebook 01 (read straight from the course GitHub repository)
CSV_URL = ("https://raw.githubusercontent.com/AdityaR28/"
           "OSS4008A---Introduction-to-Remote-Sensing-and-Applications/main/"
           "students/nifemi_ogunmeru/data/canada_airports_clean.csv")

# Ontario aerial imagery (~20 cm per pixel), served as ready-made map tiles. ArcGIS orders tiles as z/y/x.
ONTARIO_TILES = ("https://ws.lioservices.lrc.gov.on.ca/arcgis2/rest/services/"
                 "LIO_Imagery/Ontario_Imagery_Web_Map_Service/MapServer/tile/{z}/{y}/{x}")
ONTARIO_ATTR = "Contains information licensed under the Open Government Licence – Ontario"

# Colour, dot size and label for each airport type (same colours as notebook 01)
STYLES = {
    "large_airport":  {"color": "#d7191c", "radius": 7, "label": "Large airports"},
    "medium_airport": {"color": "#fdae61", "radius": 5, "label": "Medium airports"},
    "small_airport":  {"color": "#2b83ba", "radius": 3, "label": "Small airports"},
    "seaplane_base":  {"color": "#5ab4ac", "radius": 3, "label": "Seaplane bases"},
    "heliport":       {"color": "#7b3294", "radius": 3, "label": "Heliports"},
}

PROVINCES = {"AB": "Alberta", "BC": "British Columbia", "MB": "Manitoba", "NB": "New Brunswick",
             "NL": "Newfoundland and Labrador", "NS": "Nova Scotia", "NT": "Northwest Territories",
             "NU": "Nunavut", "ON": "Ontario", "PE": "Prince Edward Island", "QC": "Quebec",
             "SK": "Saskatchewan", "YT": "Yukon"}

# Bottom buttons: the class projects from the course document (number, short label)
PROJECTS = [(1, "Homes / WUI"), (2, "Factories"), (3, "Oil & pipes"), (4, "Railways"),
            (5, "Hospitals"), (8, "Airports ✈"), (9, "Ships & ports"), (11, "Water"),
            (12, "Real estate"), (13, "Farms"), (22, "Disasters"), (23, "Wildfires"), (24, "Floods")]
MY_PROJECT = 8

CANADA_VIEW = {"center": [56.0, -96.0], "zoom": 4}


@st.cache_data                     # download the CSV once, not on every click
def load_airports():
    return pd.read_csv(CSV_URL)


airports = load_airports()

# st.session_state keeps values between reruns (Streamlit re-runs this whole file after every click)
if "view" not in st.session_state:
    st.session_state.view = dict(CANADA_VIEW)
GREETING = {"role": "assistant", "content": "Hi! Ask me about Canada's airports, or type **help**."}
DEFAULT_CONF = 0.7                      # plane-detection confidence shown by default
if "chat" not in st.session_state:
    st.session_state.chat = [GREETING]
if "conf" not in st.session_state:
    st.session_state.conf = DEFAULT_CONF
if "map_version" not in st.session_state:
    st.session_state.map_version = 0    # changing it gives a brand-new map (drawn boxes removed)
if "last_drawings" not in st.session_state:
    st.session_state.last_drawings = []


def reset_dashboard():
    """Back to the start: Canada view, no drawn boxes or detections, default confidence, fresh chat."""
    st.session_state.view = dict(CANADA_VIEW)
    st.session_state.chat = [GREETING, {"role": "assistant",
                             "content": "Dashboard reset: map, drawn boxes, detections and chat cleared."}]
    st.session_state.conf = DEFAULT_CONF
    st.session_state.last_drawings = []
    st.session_state.map_version += 1


# A reset asked for on the previous run (chat "reset" or the project button) is applied here, at the very
# top, before any widget is drawn: Streamlit only allows changing a widget's value before it is created.
if st.session_state.pop("reset_requested", False):
    reset_dashboard()


# ---------------------------------------------------------------- helper functions
def text(value):
    """Show missing values (NaN) as a dash instead of 'nan'."""
    return "—" if pd.isna(value) or value == "" else str(value)


def type_geojson(t):
    """All airports of one type as one compact GeoJSON layer (much smaller and faster than 2,477 separate markers).
    The properties hold the text shown in the popup ('click on pixel → information' in the draft)."""
    features = []
    for _, row in airports[airports["type"] == t].iterrows():
        elev = "—" if pd.isna(row["elevation_ft"]) else f"{int(row['elevation_ft'])} ft"
        features.append({
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [float(row["lon"]), float(row["lat"])]},   # GeoJSON = [lon, lat]
            "properties": {"name": row["name"],
                           "kind": STYLES[t]["label"][:-1],                     # "Large airports" -> "Large airport"
                           "codes": f"{text(row['iata_code'])} / {text(row['icao_code'])}",
                           "place": f"{text(row['municipality'])}, {row['province']}",
                           "elev": elev,
                           "sched": text(row["scheduled_service"])},
        })
    return {"type": "FeatureCollection", "features": features}


@st.cache_resource      # build the map ONCE (a clean master copy); moving the view is done by st_folium(center=..., zoom=...)
def build_map():
    """Basemap, Ontario aerial layer, one layer per airport type, draw tool."""
    m = folium.Map(location=CANADA_VIEW["center"], zoom_start=CANADA_VIEW["zoom"], max_zoom=20,
                   control_scale=True, prefer_canvas=True)   # canvas draws thousands of dots much faster

    # Aerial imagery first, so the airport dots sit on top.
    # min_zoom=12: the layer stays switched on but only loads when you zoom in to airport level.
    folium.TileLayer(tiles=ONTARIO_TILES, attr=ONTARIO_ATTR, name="Ontario aerial imagery (~20 cm)",
                     overlay=True, show=True, min_zoom=12, max_zoom=20).add_to(m)

    # Airport layers, drawn small → large so large airports end up on top
    for t in reversed(list(STYLES)):
        s = STYLES[t]
        folium.GeoJson(
            type_geojson(t),
            name=s["label"],
            show=(t not in ["heliport", "seaplane_base"]),
            marker=folium.CircleMarker(radius=s["radius"], fill=True),
            style_function=lambda f, c=s["color"]: {"color": c, "fillColor": c,
                                                     "fillOpacity": 0.85, "weight": 1},
            tooltip=folium.GeoJsonTooltip(fields=["name"], labels=False),
            popup=folium.GeoJsonPopup(fields=["name", "kind", "codes", "place", "elev", "sched"],
                                      aliases=["Name", "Type", "IATA / ICAO", "Location",
                                               "Elevation", "Scheduled service"]),
        ).add_to(m)

    # Rectangle-only draw tool ("drawing a box" in the draft)
    Draw(draw_options={"rectangle": True, "polyline": False, "polygon": False, "circle": False,
                       "marker": False, "circlemarker": False},
         edit_options={"edit": False}).add_to(m)
    folium.LayerControl(collapsed=True).add_to(m)
    return m


def airports_in_box(drawing):
    """Return the airports inside a rectangle drawn on the map (GeoJSON coordinates are [lon, lat])."""
    coords = drawing["geometry"]["coordinates"][0]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    inside = (airports["lon"].between(min(lons), max(lons)) &
              airports["lat"].between(min(lats), max(lats)))
    return airports[inside]


def answer(question):
    """Simple rule-based assistant (a placeholder for the AI model planned for Progress 3)."""
    q = question.strip()
    ql = q.lower()

    if ql in ["help", "?"]:
        return ("Try:\n- **YOW** (or any airport code) – zoom to that airport\n"
                "- **how many airports in ON** (province code in capitals, or the full name)\n"
                "- **how many heliports** / large / medium / small / seaplane\n"
                "- **reset** – reset the whole dashboard (map, boxes, detections, chat)\n"
                "- Draw a box on the map (square tool) for an area summary\n"
                "- **Draw a box around an Ontario airport apron** (about 1 km or less) to **detect planes** with the AI model")

    if ql == "reset":
        st.session_state.reset_requested = True      # applied at the top of the next run
        return "Resetting the dashboard…"

    if "how many" in ql:
        # Province (code or full name)
        for code, name in PROVINCES.items():
            if code in q.replace("?", " ").split() or name.lower() in ql:   # codes must be capitals (ON, not "on")
                n = int((airports["province"] == code).sum())
                return f"**{n}** active facilities in {name} (airports, heliports and seaplane bases)."
        # Facility type
        for key, t in [("heliport", "heliport"), ("seaplane", "seaplane_base"), ("large", "large_airport"),
                       ("medium", "medium_airport"), ("small", "small_airport")]:
            if key in ql:
                n = int((airports["type"] == t).sum())
                return f"**{n}** {STYLES[t]['label'].lower()} in Canada."
        return f"**{len(airports):,}** active facilities in Canada."

    # Airport codes: 3 letters (IATA, e.g. YOW) or 4 (ICAO, e.g. CYOW)
    for word in q.upper().replace("?", " ").replace(",", " ").split():
        match = airports[(airports["iata_code"] == word) | (airports["icao_code"] == word) |
                         (airports["ident"] == word)]
        if len(match) > 0:
            a = match.iloc[0]
            st.session_state.view = {"center": [float(a["lat"]), float(a["lon"])], "zoom": 16}
            extra = (" The aerial layer is on, so zoom in to see parked planes."
                     if a["province"] == "ON" else
                     " Sharp aerial imagery is only available in Ontario for now.")
            return f"Zoomed to **{a['name']}** ({text(a['iata_code'])}), {text(a['municipality'])}, {a['province']}.{extra}"

    return "I can't answer that yet. Type **help** to see what I understand. (The AI model arrives in Progress 3.)"


# ---------------------------------------------------------------- plane detection (AI model from notebook 04)
MODEL_PATH = "/content/drive/MyDrive/oss4008a_planes/plane_detector.pt"   # saved by notebook 04
DET_ZOOM = 19              # same imagery zoom as the training chips (~0.21 m per pixel)
CHIP_TILES = 4             # 4 x 4 tiles = 1024 x 1024 px chips, same as training
MAX_CHIPS = 36             # about 1 km x 1 km; larger boxes would take too long
MIN_SCORE = 0.3            # keep weaker detections too, so the confidence slider can filter without re-running
ONTARIO = (-95.2, 41.6, -74.3, 57.0)   # rough west, south, east, north of Ontario (where the 20 cm imagery is)


def lonlat_to_tile(lon, lat, z):
    """Which web-map tile (x, y) contains this point at zoom z (Web Mercator)."""
    n = 2 ** z
    lat_r = math.radians(lat)
    return (int((lon + 180.0) / 360.0 * n),
            int((1.0 - math.log(math.tan(lat_r) + 1.0 / math.cos(lat_r)) / math.pi) / 2.0 * n))


def pixel_to_lonlat(gx, gy, z):
    """Longitude/latitude of a 'global pixel' (tile number x 256 + pixel inside the tile) at zoom z."""
    n = 256 * 2 ** z
    return gx / n * 360.0 - 180.0, math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * gy / n))))


def chips_for_box(w, s, e, n, z=DET_ZOOM, ct=CHIP_TILES):
    """Top-left tiles of the chips (on the same fixed grid as training) that cover the box."""
    x_min, y_min = lonlat_to_tile(w, n, z)
    x_max, y_max = lonlat_to_tile(e, s, z)
    return [(cx * ct, cy * ct) for cx in range(x_min // ct, x_max // ct + 1)
                               for cy in range(y_min // ct, y_max // ct + 1)]


def fetch_tile(z, x, y):
    """Download one Ontario imagery tile (bytes), or None if it is missing."""
    try:
        r = requests.get(ONTARIO_TILES.format(z=z, x=x, y=y), timeout=20)
        if r.status_code == 200 and r.headers.get("Content-Type", "").startswith("image"):
            return r.content
    except requests.RequestException:
        pass
    return None


def build_chip(x0, y0, z=DET_ZOOM, ct=CHIP_TILES):
    """Stitch ct x ct tiles into one 1024 px chip (grey where a tile is missing). Returns (image, tiles found)."""
    chip = Image.new("RGB", (256 * ct, 256 * ct), (128, 128, 128))
    found = 0
    for i in range(ct):
        for j in range(ct):
            data = fetch_tile(z, x0 + i, y0 + j)
            if data:
                try:
                    chip.paste(Image.open(io.BytesIO(data)).convert("RGB"), (i * 256, j * 256))
                    found += 1
                except Exception:
                    pass
    return chip, found


def merge_split_planes(dets, gap=6):
    """A plane lying across the border of two chips is found twice, as two halves. Join boxes from
    DIFFERENT chips that touch (within `gap` pixels) into one box. dets: dicts with 'box' in global
    pixels [x0, y0, x1, y1], 'score' and 'chip'."""
    parent = list(range(len(dets)))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for i in range(len(dets)):
        for j in range(i + 1, len(dets)):
            a, b = dets[i], dets[j]
            if a["chip"] == b["chip"]:
                continue
            A, B = a["box"], b["box"]
            if A[0] - gap <= B[2] and B[0] - gap <= A[2] and A[1] - gap <= B[3] and B[1] - gap <= A[3]:
                parent[root(i)] = root(j)
    groups = {}
    for i, d in enumerate(dets):
        groups.setdefault(root(i), []).append(d)
    merged = []
    for g in groups.values():
        boxes = np.array([d["box"] for d in g])
        merged.append({"box": [float(boxes[:, 0].min()), float(boxes[:, 1].min()),
                               float(boxes[:, 2].max()), float(boxes[:, 3].max())],
                       "score": max(d["score"] for d in g)})
    return merged


@st.cache_resource(show_spinner="Loading the plane detector…")
def load_model():
    """The Faster R-CNN plane detector trained in notebook 04 (same architecture, weights from Google Drive)."""
    import torch
    from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    model = fasterrcnn_resnet50_fpn_v2(weights=None, weights_backbone=None, min_size=1024, max_size=1024)
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, 2)
    model.load_state_dict(torch.load(MODEL_PATH, map_location="cpu"))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    return model.to(device).eval(), device


@st.cache_data(show_spinner=False)          # the same box is never processed twice
def detect_planes(w, s, e, n):
    """Find planes inside the box (west, south, east, north). Returns (planes, chips used, message).
    Each plane: dict with south, west, north, east (degrees) and score (0-1)."""
    if not (ONTARIO[0] <= w and e <= ONTARIO[2] and ONTARIO[1] <= s and n <= ONTARIO[3]):
        return [], 0, "Plane detection works in **Ontario** only (where open 20 cm aerial imagery exists)."
    chips = chips_for_box(w, s, e, n)
    if len(chips) > MAX_CHIPS:
        return [], len(chips), (f"That box is too big ({len(chips)} image chips; the limit is {MAX_CHIPS}). "
                                "Zoom in to an airport and draw a box about **1 km across** or smaller.")
    import os
    if not os.path.exists(MODEL_PATH):
        return [], 0, "Model file not found in Google Drive: run notebook 04 first."
    import torch
    model, device = load_model()

    with ThreadPoolExecutor(max_workers=8) as pool:          # download the chips in parallel
        built = list(pool.map(lambda c: build_chip(*c), chips))

    dets = []
    with torch.no_grad():
        for (x0, y0), (img, found) in zip(chips, built):
            if found == 0:                                   # no imagery for this chip at all
                continue
            t = torch.from_numpy(np.array(img)).permute(2, 0, 1).float().div(255).to(device)
            out = model([t])[0]
            for (bx0, by0, bx1, by1), sc in zip(out["boxes"].cpu().tolist(), out["scores"].cpu().tolist()):
                if sc >= MIN_SCORE:                          # chip pixels -> global pixels
                    dets.append({"box": [x0 * 256 + bx0, y0 * 256 + by0, x0 * 256 + bx1, y0 * 256 + by1],
                                 "score": sc, "chip": (x0, y0)})

    planes = []
    for d in merge_split_planes(dets):
        gx0, gy0, gx1, gy1 = d["box"]
        west, north = pixel_to_lonlat(gx0, gy0, DET_ZOOM)
        east, south = pixel_to_lonlat(gx1, gy1, DET_ZOOM)
        lon_c, lat_c = (west + east) / 2, (south + north) / 2
        if w <= lon_c <= e and s <= lat_c <= n:              # keep only planes whose centre is inside the box
            planes.append({"south": south, "west": west, "north": north, "east": east, "score": d["score"]})
    if not built or all(f == 0 for _, f in built):
        return [], len(chips), "No aerial imagery could be downloaded for this box. Try again in a moment."
    return planes, len(chips), ""


# ---------------------------------------------------------------- page layout
# Streamlit greys out the page while it works ("stale" elements). This style rule keeps everything fully visible.
st.markdown("<style>[data-stale='true'], .stale-element {opacity: 1 !important; filter: none !important;}</style>",
            unsafe_allow_html=True)

st.markdown("### 🛰️ OSS4008A Remote Sensing Dashboard  ·  Project 8: Airports and airplanes in Canada")
left, right = st.columns([1, 2.2], gap="medium")
st.divider()
bottom = st.container()     # created now so it appears at the bottom; filled first (see step 1)

# Order matters for speed: buttons and chat are handled BEFORE the map is drawn,
# so the map already shows the new view and the page only runs once per action.

# 1) BOTTOM: one button per class project (others will be connected when the projects merge)
with bottom:
    cols = st.columns(len(PROJECTS))
    for col, (num, label) in zip(cols, PROJECTS):
        mine = (num == MY_PROJECT)
        if col.button(label, key=f"project_{num}", disabled=not mine,
                      type="primary" if mine else "secondary",
                      help=f"Project {num}" + ("" if mine else " (connected when projects merge)")):
            st.session_state.reset_requested = True        # pressing the project button reloads the whole dashboard
            st.rerun()

# 2) LEFT: assistant panel. Areas are reserved now and filled after the map (step 4).
with left:
    st.markdown("#### 🤖 AI assistant")
    # A form only sends the value when "Apply" is clicked, so quick changes don't pile up and jam the map
    with st.form("confidence_form", border=False):
        st.select_slider("Plane detection confidence", options=[0.5, 0.6, 0.7, 0.8, 0.9], key="conf",
                         help="Only planes the model is at least this sure about are shown. "
                              "Lower = more planes found, but more false alarms.")
        st.form_submit_button("Apply")
    conf = st.session_state.conf
    summary_area = st.container()
    chat_area = st.container(height=230)
    prompt = st.chat_input("Ask about airports… (try: YOW)")
    if prompt:
        st.session_state.chat.append({"role": "user", "content": prompt})
        st.session_state.chat.append({"role": "assistant", "content": answer(prompt)})
        if st.session_state.get("reset_requested"):
            st.rerun()

# 3) RIGHT: the interactive map. The map itself is built once (cached); only centre and zoom change.
# The boxes drawn on the map come back from the map component. Streamlit keeps the component's last value
# in st.session_state[MAP_KEY], so the box is known BEFORE the map is drawn and the detected planes can be
# drawn on this same run.
MAP_KEY = f"map_{st.session_state.map_version}"
previous = st.session_state.get(MAP_KEY)
if isinstance(previous, dict) and "all_drawings" in previous:
    drawings = previous.get("all_drawings") or []          # boxes currently on the map (deleted ones are gone)
else:
    drawings = st.session_state.last_drawings
drawing = drawings[-1] if drawings else None               # use the most recent box
is_box = bool(drawing) and drawing.get("geometry", {}).get("type") == "Polygon"

planes, n_chips, det_msg = [], 0, ""
if is_box:
    coords = drawing["geometry"]["coordinates"][0]                     # GeoJSON: [lon, lat] pairs
    box = (min(c[0] for c in coords), min(c[1] for c in coords), max(c[0] for c in coords), max(c[1] for c in coords))
    with summary_area:
        with st.spinner("🛰️ Downloading imagery and detecting planes…"):
            planes, n_chips, det_msg = detect_planes(*box)
shown = [p for p in planes if p["score"] >= conf]

# Detected planes as a separate layer: added to the map without rebuilding it
plane_layer = folium.FeatureGroup(name="Detected planes (AI)")
for p in shown:
    folium.Rectangle(bounds=[[p["south"], p["west"]], [p["north"], p["east"]]], color="#ff2d55", weight=2,
                     fill=False, tooltip=f"plane · confidence {p['score']:.2f}").add_to(plane_layer)

with right:
    view = st.session_state.view
    # returned_objects keeps the app fast: only the drawn box is sent back, not every pan or zoom
    # Draw a fresh COPY of the master map: drawing a map changes it slightly, so reusing the same object
    # after a page refresh would produce broken map code (the map collapsed to a thin strip).
    map_data = st_folium(copy.deepcopy(build_map()), center=view["center"], zoom=view["zoom"],
                         height=620, use_container_width=True,
                         returned_objects=["all_drawings"], feature_group_to_add=plane_layer, key=MAP_KEY)

# Safety net: if the box did not reach us before the map was drawn, run the page once more
new_drawings = (map_data or {}).get("all_drawings") or []
if new_drawings != drawings:
    st.session_state.last_drawings = new_drawings    # remember them, so the next run uses them (no endless reruns)
    st.rerun()

# 4) LEFT (continued): area summary and chat history
with summary_area:
    if is_box:
        # Summary of the box the user drew
        sel = airports_in_box(drawing)
        st.markdown("**Selected area**")
        c1, c2 = st.columns(2)
        c1.metric("Facilities", len(sel))
        c2.metric("Planes detected (AI)", len(shown) if not det_msg else "—",
                  help="Faster R-CNN model fine-tuned on labelled Ontario airport imagery (notebook 04)")
        if det_msg:
            st.info(det_msg)
        elif shown:
            st.caption(f"{len(shown)} planes in {n_chips} image chips · confidence ≥ {conf:.2f} · "
                       f"average {np.mean([p['score'] for p in shown]):.2f}. Red boxes on the map; hover for confidence.")
        else:
            st.caption(f"No planes found above confidence {conf:.2f} in {n_chips} image chips.")
        if len(sel) > 0:
            counts = sel["type"].map(lambda t: STYLES[t]["label"]).value_counts()
            st.bar_chart(counts, horizontal=True, height=180)
            main = sel[sel["type"].isin(["large_airport", "medium_airport"])]
            if len(main) > 0:
                st.dataframe(main[["name", "iata_code", "municipality", "province"]].fillna("—"),
                             hide_index=True, height=160)
    else:
        # Canada-wide overview until a box is drawn
        st.markdown("**All of Canada**: draw a box on the map for an area summary")
        c1, c2 = st.columns(2)
        c1.metric("Facilities", f"{len(airports):,}")
        c2.metric("Large airports", int((airports["type"] == "large_airport").sum()))
        counts = airports["type"].map(lambda t: STYLES[t]["label"]).value_counts()
        st.bar_chart(counts, horizontal=True, height=180)

with chat_area:
    for msg in st.session_state.chat:
        st.chat_message(msg["role"]).markdown(msg["content"])

In [ ]:
# Cell 3 – Check that app.py is the dashboard (not an old test file) and has no Python syntax errors
src = open('app.py', encoding='utf-8').read()
assert 'OSS4008A shared dashboard' in src, 'app.py is not the dashboard – rerun Cell 2'
compile(src, 'app.py', 'exec')            # raises an error here if the code has a syntax mistake
print(f'app.py OK ✅  ({len(src.splitlines())} lines)')

In [ ]:
# Cell 4 – (Re)start the Streamlit server in the background and wait until it answers
import time, urllib.request

!pkill -f "streamlit run" || true          # stop any copy that is already running (avoids "port not available")
time.sleep(2)
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit_log.txt 2>&1 &

ok = False
for _ in range(30):                        # check once per second, for up to 30 seconds
    time.sleep(1)
    try:
        ok = urllib.request.urlopen('http://localhost:8501/_stcore/health').read().decode() == 'ok'
        if ok:
            break
    except Exception:
        pass                               # server not ready yet

if ok:
    print('Streamlit server is running ✅')
else:
    print('❌ Streamlit did not start. Server log:')
    print(open('streamlit_log.txt').read())

In [ ]:
# Cell 5 – Public link to the app through a free Cloudflare quick tunnel (no account needed)
import os, re

if not os.path.exists('cloudflared'):      # download the tunnel program once per session
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x cloudflared

!pkill -f "cloudflared tunnel" || true     # close any old tunnel
!nohup ./cloudflared tunnel --url http://localhost:8501 --no-autoupdate > tunnel_log.txt 2>&1 &

url = None
for _ in range(30):                        # wait up to 30 seconds for the link to appear in the log
    time.sleep(1)
    found = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel_log.txt').read())
    if found:
        url = found[0]
        break

print('Open the dashboard here 👉', url if url else 'no link yet – run this cell again')
print('If the page shows an error at first, wait 15–20 seconds and refresh.')